# ⚙️ Feature Engineering
**ROGII — Wellbore Geology Prediction**

> Generate domain-driven features from raw drilling and LWD data:
> rolling statistics, lag features, drilling mechanics (MSE), petrophysical ratios (Vsh), and directional survey encodings.

---
**Author:** Md Ashraf | M.Sc (Tech) Applied Geophysics, IIT (ISM) Dhanbad

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from src import preprocessing, feature_engineering

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

In [ ]:
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

DATA_DIR   = '../' + cfg['paths']['raw_dir']
PROC_DIR   = '../' + cfg['paths']['processed_dir']
TARGET     = cfg['data']['target_column']
WELL_COL   = cfg['data']['well_id_column']
DEPTH_COL  = cfg['data']['depth_column']
FEAT_CFG   = cfg['features']

Path(PROC_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded ✅')

## 1. Load and Clean Data

In [ ]:
try:
    train, test, sample_sub = preprocessing.load_data(
        DATA_DIR,
        cfg['data']['train_file'],
        cfg['data']['test_file'],
        cfg['data']['sample_submission_file'],
    )
except FileNotFoundError:
    print('⚠️  Generating synthetic demo data...')
    np.random.seed(42)
    n_wells, n_depth = 5, 500
    rows = []
    for w in range(n_wells):
        for d in range(n_depth):
            rows.append({
                'well_id': f'WELL_{w:02d}',
                'md': 2000 + d * 2,
                'tvd': 1800 + d * 0.5 + np.random.randn() * 5,
                'inclination': 85 + np.random.randn() * 2,
                'azimuth': 135 + np.random.randn() * 3,
                'rop': np.random.lognormal(2.5, 0.4),
                'wob': np.random.lognormal(3.2, 0.3),
                'rpm': np.random.normal(120, 15),
                'torque': np.random.lognormal(4.0, 0.5),
                'flow_rate': np.random.normal(400, 30),
                'ecd': np.random.normal(1.35, 0.05),
                'gr': np.random.lognormal(4.2, 0.5),
                'resistivity': np.random.lognormal(2.0, 1.0),
                'neutron': np.random.normal(0.25, 0.05),
                'density': np.random.normal(2.4, 0.1),
                'sonic': np.random.normal(90, 10),
                'formation': np.random.uniform(0, 10),
            })
    train = pd.DataFrame(rows)
    test  = train.sample(frac=0.2, random_state=42).drop(columns=['formation'])
    sample_sub = pd.DataFrame({'id': range(len(test)), 'formation': 0})

# Sort by well + depth (critical for rolling/lag features)
train = train.sort_values([WELL_COL, DEPTH_COL]).reset_index(drop=True)
test  = test.sort_values([WELL_COL, DEPTH_COL]).reset_index(drop=True)
print(f'Train: {train.shape} | Test: {test.shape}')

In [ ]:
# Handle missing values (interpolate within each well)
train = preprocessing.handle_missing_values(train, strategy='interpolate', group_col=WELL_COL)
test  = preprocessing.handle_missing_values(test,  strategy='interpolate', group_col=WELL_COL)
print('Missing values handled ✅')

## 2. Rolling Statistics

In [ ]:
drilling_cols = [c for c in FEAT_CFG.get('drilling_features', []) if c in train.columns]
petro_cols    = [c for c in FEAT_CFG.get('petrophysical_features', []) if c in train.columns]
rolling_windows = FEAT_CFG.get('rolling_windows', [3, 5, 10])

all_base_features = drilling_cols + petro_cols
print(f'Base features: {all_base_features}')
print(f'Rolling windows: {rolling_windows}')

In [ ]:
train = feature_engineering.add_rolling_features(
    train, all_base_features, rolling_windows, group_col=WELL_COL
)
test  = feature_engineering.add_rolling_features(
    test, all_base_features, rolling_windows, group_col=WELL_COL
)
print(f'After rolling features — Train: {train.shape}')

## 3. Lag Features

In [ ]:
lag_steps = FEAT_CFG.get('lag_steps', [1, 2, 3])

train = feature_engineering.add_lag_features(
    train, all_base_features, lag_steps, group_col=WELL_COL
)
test  = feature_engineering.add_lag_features(
    test, all_base_features, lag_steps, group_col=WELL_COL
)
print(f'After lag features — Train: {train.shape}')

## 4. Depth Features

In [ ]:
train = feature_engineering.add_depth_features(train, depth_col=DEPTH_COL, well_col=WELL_COL)
test  = feature_engineering.add_depth_features(test,  depth_col=DEPTH_COL, well_col=WELL_COL)
print('Depth features added ✅')
display(train[['md', 'tvd', 'depth_normalized', 'depth_from_well_top', 'tvd_md_ratio']].head())

## 5. Drilling Mechanics Features (MSE)

In [ ]:
train = feature_engineering.add_drilling_features(train)
test  = feature_engineering.add_drilling_features(test)

new_drilling_cols = [c for c in ['mse', 'rev_per_ft', 'wob_rpm_ratio', 'specific_torque'] if c in train.columns]
print(f'New drilling features: {new_drilling_cols}')
if new_drilling_cols:
    display(train[new_drilling_cols].describe().round(3))

## 6. Petrophysical Features (Vsh, AI)

In [ ]:
train = feature_engineering.add_petrophysical_features(train)
test  = feature_engineering.add_petrophysical_features(test)

new_petro_cols = [c for c in ['vsh_gr', 'log_resistivity', 'neutron_density_separation', 'acoustic_impedance'] if c in train.columns]
print(f'New petrophysical features: {new_petro_cols}')
if new_petro_cols:
    display(train[new_petro_cols].describe().round(3))

## 7. Directional Survey Features

In [ ]:
train = feature_engineering.add_directional_features(train)
test  = feature_engineering.add_directional_features(test)

dir_cols = [c for c in ['sin_inclination', 'cos_inclination', 'sin_azimuth', 'cos_azimuth'] if c in train.columns]
print(f'Directional features: {dir_cols}')

## 8. Final Feature Set

In [ ]:
exclude_cols = [TARGET, WELL_COL, 'id']
feature_cols = [c for c in train.columns if c not in exclude_cols
                and train[c].dtype in [np.float32, np.float64, np.int32, np.int64]]

print(f'Total features: {len(feature_cols)}')
print(feature_cols[:20], '...')

In [ ]:
# Fill remaining NaN from lag/rolling at depth boundaries
train[feature_cols] = train[feature_cols].fillna(train[feature_cols].median())
test[feature_cols]  = test[feature_cols].fillna(train[feature_cols].median())

print('NaN fill complete ✅')
print(f'Train: {train.shape} | Remaining NaN: {train[feature_cols].isnull().sum().sum()}')

In [ ]:
# Save processed datasets
train.to_parquet(f'{PROC_DIR}/train_features.parquet', index=False)
test.to_parquet(f'{PROC_DIR}/test_features.parquet', index=False)

# Save feature list
import json
with open(f'{PROC_DIR}/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print(f'Processed data saved to {PROC_DIR} ✅')

---
## ✅ Summary

| Feature Group | Count |
|---|---|
| Rolling statistics | ~4 stats × windows × base features |
| Lag features | lag_steps × base features |
| Depth features | 4 |
| Drilling mechanics | 4 (MSE, rev/ft, WOB/RPM, specific torque) |
| Petrophysical | 4 (Vsh, log Rt, N-D separation, AI) |
| Directional | 4–5 (sin/cos encoded) |

**Next step → `03_baseline_lightgbm.ipynb`**